# 4. Limpieza de datos

La limpieza de datos consiste en revisar y preparar un dataset para que sus valores tengan un formato coherente y puedan utilizarse correctamente en el análisis.

Antes de modificar los datos, es importante inspeccionarlos para identificar los problemas reales. No debemos eliminar ni transformar valores sin comprobar primero qué contienen y qué significado tienen.

## 4.1 Valores nulos

Los valores nulos (`NaN`) representan datos ausentes. Podemos detectarlos, contar cuántos hay y decidir cómo tratarlos.

### Detectar y contar valores nulos

```python
df.isna().sum()
# Cuenta los valores nulos de cada columna del DataFrame.
```

### Eliminar filas con valores nulos

```python
df = df.dropna(subset=["precio"])
# Elimina únicamente las filas donde la columna precio tiene un valor nulo.
```

`dropna()` devuelve un DataFrame sin las filas seleccionadas. Al asignarlo de nuevo a `df`, conservamos el resultado.

### Rellenar valores nulos

Podemos sustituir los valores nulos por una medida estadística o por un valor concreto.

```python
df["precio"] = df["precio"].fillna(df["precio"].mean())
# Rellena los nulos de precio con la media de la columna.
# mean() calcula la media ignorando los valores NaN.
```

```python
df["edad"] = df["edad"].fillna(df["edad"].median())
# Rellena los nulos de edad con la mediana de la columna.
```

```python
df["precio"] = df["precio"].fillna(0)
# Sustituye los valores nulos de precio por 0.
```

**Importante:** no debemos sustituir automáticamente todos los nulos por cero o por la media. La decisión depende del significado de la columna y del análisis que vayamos a realizar.

## 4.2 Valores duplicados

Los duplicados son registros que aparecen más de una vez. Podemos detectarlos y, si procede, eliminarlos.

```python
df.duplicated().sum()
# Cuenta cuántas filas están duplicadas.
```

```python
df = df.drop_duplicates()
# Elimina las filas duplicadas y conserva la primera aparición.
```

**Nota:** `drop_duplicates()` devuelve un nuevo DataFrame. Para conservar el resultado debemos asignarlo a `df` o a otra variable.

Antes de eliminar duplicados, debemos comprobar que realmente son registros repetidos y no operaciones legítimas que comparten los mismos valores.

## 4.3 Limpieza de columnas numéricas

En ocasiones, una columna que debería contener números llega como texto (`object` o `string`). Puede incluir espacios, símbolos de moneda, separadores decimales o caracteres que impiden convertirla directamente a un tipo numérico.

El proceso consiste en inspeccionar la columna, localizar los valores problemáticos, limpiarlos y convertirlos.

### Inspeccionar la columna

```python
df["col"].dtype
# Devuelve el tipo de dato de la columna.

df["col"].isna().sum()
# Cuenta los valores nulos.
```

### Detectar valores formados únicamente por dígitos

```python
df["col"].astype(str).str.isnumeric().value_counts()
# Cuenta cuántos valores están formados únicamente por dígitos.
# True: por ejemplo, "12".
# False: por ejemplo, "12a".
```

**Importante:** `isnumeric()` detecta cadenas formadas por caracteres numéricos, pero no identifica como numéricos todos los formatos válidos para una columna de importes. Por ejemplo, valores con decimales, signos o separadores pueden devolver `False`. Por eso esta comprobación sirve como inspección inicial, no como validación definitiva.

### Localizar valores problemáticos

```python
df.loc[~df["col"].astype(str).str.isnumeric(), "col"]
# Muestra los valores que no están formados únicamente por dígitos.
```

El operador `~` invierte la condición: selecciona los valores para los que `isnumeric()` devuelve `False`.

### Detectar símbolos y caracteres extraños

```python
df["col"].astype(str).str.extract(r"([^\d\.,])").value_counts()
# Extrae y cuenta caracteres que no sean dígitos, punto o coma.
```

Esta comprobación permite localizar caracteres como símbolos de moneda o letras. El patrón admite puntos y comas porque pueden utilizarse como separadores numéricos.

## 4.4 Limpiar y transformar columnas numéricas

Una vez inspeccionados los valores, podemos normalizar el contenido de la columna.

```python
df["col"] = (
    df["col"]
    .astype("string")  # Convierte los valores a texto y conserva los nulos como <NA>.
    .str.strip()       # Elimina espacios al principio y al final.
)
```

Utilizamos el tipo `string` de Pandas para conservar los valores ausentes como `<NA>`, en lugar de convertirlos en el texto `"nan"`.

## 4.5 Eliminar símbolos y palabras detectadas

Si durante la inspección hemos encontrado símbolos o palabras que deben eliminarse, podemos utilizar `str.replace()`.

```python
df["col"] = (
    df["col"]
    .str.replace("€", "", regex=False)
    .str.replace("EUR", "", regex=False)
    .str.replace("euros", "", regex=False)
)
```

Este ejemplo elimina los textos indicados. Solo debemos incluir los símbolos o palabras que hayamos detectado en los datos.

`regex=False` indica que el texto que buscamos debe interpretarse literalmente y no como una expresión regular.

## 4.6 Normalizar separadores numéricos

Si los datos utilizan el formato europeo, podemos encontrar el punto como separador de miles y la coma como separador decimal.

Por ejemplo:

- `1.250,50` representa mil doscientos cincuenta con cincuenta.
- `850,75` representa ochocientos cincuenta con setenta y cinco.

Podemos normalizar estos separadores antes de convertir la columna:

```python
df["col"] = (
    df["col"]
    .str.replace(".", "", regex=False)   # Elimina el separador de miles.
    .str.replace(",", ".", regex=False)  # Convierte la coma decimal en punto.
)
```

**Importante:** esta transformación solo es correcta si hemos comprobado que los datos siguen el formato europeo. No debemos aplicarla a ciegas a columnas con formatos diferentes.

## 4.7 Conversión a numérico

Una vez limpiado el texto, convertimos la columna a un tipo numérico.

```python
df["col"] = pd.to_numeric(df["col"], errors="coerce")
```

- `pd.to_numeric()` intenta convertir los valores a números.
- `errors="coerce"` convierte en `NaN` los valores que no se puedan interpretar como números.

Después de la conversión, conviene revisar cuántos valores han quedado como nulos para comprobar si se han producido conversiones fallidas.

## 4.8 Convertir a entero (opcional)

Si sabemos que los valores deben ser números enteros, podemos utilizar el tipo `Int64` de Pandas.

```python
df["col"] = df["col"].astype("Int64")
```

`Int64` permite almacenar valores enteros y también valores nulos (`<NA>`).

Podemos utilizarlo, por ejemplo, en columnas como:

- `user_id`, si el identificador es numérico.
- `order_id`, si el identificador es numérico.
- Número de compras.
- Edad, si se registra en años completos.

**Importante:** no todos los identificadores deben convertirse a números. Algunos contienen letras, guiones o ceros iniciales que deben conservarse. En esos casos es preferible utilizar texto.

## 4.9 Validación final de columnas numéricas

Después de limpiar y convertir una columna, debemos comprobar que el resultado es correcto.

```python
df["col"].isna().sum()
# Cuenta los valores nulos.

df["col"].describe()
# Muestra estadísticas descriptivas de la columna.

df["col"].dtype
# Comprueba el tipo de dato final.
```

La validación permite comprobar si la columna tiene el tipo esperado, si han aparecido nuevos nulos y si sus valores presentan estadísticas coherentes.

## 4.10 Limpieza de fechas (str → datetime)

Las fechas pueden llegar como texto y presentar distintos formatos. Para trabajar correctamente con ellas, debemos convertirlas al tipo `datetime`.

### 4.10.1 Inspección inicial

```python
df["fecha"].dtype
# Comprueba el tipo de dato actual.

df["fecha"].isna().sum()
# Cuenta los valores nulos.

df["fecha"].head()
# Muestra algunos valores para inspeccionar sus formatos.
```

### 4.10.2 Normalización del texto

Antes de convertir las fechas, podemos eliminar espacios y normalizar algunos valores que representan datos ausentes.

```python
df["fecha"] = (
    df["fecha"]
    .astype("string")
    .str.strip()
    .replace(["None", "nan", "NaT"], pd.NA)
    .str.replace("/", "-", regex=False)
)
```

Este proceso:

1. Convierte los valores a texto conservando los nulos.
2. Elimina espacios al principio y al final.
3. Sustituye las cadenas `"None"`, `"nan"` y `"NaT"` por valores ausentes.
4. Unifica el separador `/` mediante `-`.

La normalización del separador no garantiza por sí sola que todas las fechas tengan un formato válido; eso se comprobará en la conversión.

### 4.10.3 Conversión segura a datetime

```python
df["fecha"] = pd.to_datetime(
    df["fecha"],
    errors="coerce",
    dayfirst=True
)
```

- `pd.to_datetime()` convierte los valores a tipo fecha.
- `errors="coerce"` convierte los valores que no se puedan interpretar en `NaT`.
- `dayfirst=True` indica que, en fechas numéricas ambiguas, el día aparece antes que el mes.

### 4.10.4 Validación

```python
df["fecha"].isna().sum()
# Cuenta las fechas nulas o que no se han podido convertir.

df["fecha"].min()
# Devuelve la fecha más antigua.

df["fecha"].max()
# Devuelve la fecha más reciente.
```

### 4.10.5 Validaciones semánticas

Además de comprobar el formato, debemos comprobar si las fechas tienen sentido en el contexto del negocio.

Por ejemplo, si estamos analizando registros que no pueden tener fechas futuras:

```python
df.loc[df["fecha"] > pd.Timestamp.today(), "fecha"] = pd.NaT
# Marca como fecha ausente las fechas posteriores al momento actual.
```

También podemos establecer una fecha mínima razonable para nuestro caso de uso:

```python
df.loc[df["fecha"] < pd.Timestamp("2000-01-01"), "fecha"] = pd.NaT
# Marca como ausentes las fechas anteriores al 1 de enero de 2000.
```

Los límites deben definirse según el negocio y el periodo real de los datos. No debemos considerar inválida una fecha únicamente por ser antigua o futura sin conocer el contexto.

## 4.11 Limpieza de columnas categóricas

Las columnas categóricas contienen etiquetas o grupos, como categorías de producto, ciudades, países o tipos de cliente.

Es habitual encontrar diferencias causadas por espacios, mayúsculas, minúsculas o distintas formas de escribir una misma categoría.

### 4.11.1 Comprobación y normalización inicial

```python
df["categoria"] = (
    df["categoria"]
    .astype("string")
    .str.strip()
    .str.lower()
)
```

- `astype("string")` normaliza el tipo de dato y conserva los valores nulos como `<NA>`.
- `str.strip()` elimina espacios al principio y al final.
- `str.lower()` convierte el texto a minúsculas.

### 4.11.2 Normalización semántica

Después de normalizar el formato, podemos unificar etiquetas que representan la misma categoría.

```python
df["categoria"] = df["categoria"].replace({
    "basic plan": "basic",
    "basico": "basic",
    "free trial": "free"
})
```

En este ejemplo, distintas etiquetas se sustituyen por una denominación común.

Las equivalencias deben decidirse según el significado real de los valores. No debemos agrupar categorías que parezcan similares si representan conceptos distintos.

### 4.11.3 Validación

```python
df["categoria"].value_counts(dropna=False)
```

Muestra la frecuencia de cada categoría e incluye también los valores nulos gracias a `dropna=False`.

## 4.12 Limpieza de identificadores (ID)

Los identificadores permiten reconocer registros, usuarios, pedidos o productos. Antes de modificarlos, debemos comprobar su formato y la unicidad esperada.

### 4.12.1 Inspección inicial

```python
df["user_id"].dtype
# Tipo de dato.

df["user_id"].isna().sum()
# Número de valores nulos.

df["user_id"].head()
# Ejemplos de valores.

df["user_id"].nunique()
# Número de identificadores únicos, sin contar nulos por defecto.
```

### 4.12.2 Comprobar duplicados

```python
df["user_id"].duplicated().sum()
```

Devuelve cuántas apariciones duplicadas existen después de la primera. Un identificador repetido no siempre es un error: depende de si la tabla debe contener una única fila por usuario o varias filas relacionadas con el mismo usuario.

### 4.12.3 Detectar IDs no numéricos

```python
df["user_id"].astype("string").str.isnumeric().value_counts(dropna=False)
```

Comprueba qué valores están formados únicamente por dígitos.

**Nota:** `isnumeric()` devuelve `False` para identificadores que contienen letras, guiones u otros caracteres. Eso no significa necesariamente que sean inválidos: un ID puede ser alfanumérico.

### 4.12.4 Ver ejemplos problemáticos

```python
df.loc[
    ~df["user_id"].astype("string").str.isnumeric(),
    "user_id"
]
```

Permite inspeccionar los valores que no están formados únicamente por dígitos antes de decidir si necesitan alguna transformación.

## 4.13 Normalizar identificadores

Si hemos detectado espacios o prefijos que deben eliminarse, podemos normalizar el identificador.

### 4.13.1 Convertir a texto y eliminar espacios

```python
df["user_id"] = (
    df["user_id"]
    .astype("string")
    .str.strip()
)
```

Se utiliza `string` para conservar los valores nulos como `<NA>`.

### 4.13.2 Eliminar caracteres o prefijos detectados

```python
df["user_id"] = (
    df["user_id"]
    .str.replace("id_", "", regex=False)
    .str.replace("-", "", regex=False)
)
```

Este ejemplo elimina el prefijo `id_` y los guiones.

**Importante:** nunca debemos eliminar caracteres a ciegas. Primero inspeccionamos los valores y comprobamos qué formato debe tener el identificador. Los ceros iniciales, guiones o letras pueden formar parte del ID y ser necesarios para identificar correctamente un registro.

## 4.14 Conversión final del ID

El tipo de dato final dependerá de la naturaleza del identificador.

### Opción A — ID como entero

Si el identificador debe ser estrictamente numérico y no necesitamos conservar ceros iniciales:

```python
df["user_id"] = pd.to_numeric(
    df["user_id"],
    errors="coerce"
).astype("Int64")
```

`Int64` permite almacenar enteros y valores nulos.

### Opción B — ID como texto

Si el identificador puede contener letras, guiones, ceros iniciales o formatos externos como UUID:

```python
df["user_id"] = df["user_id"].astype("string")
```

En muchos sistemas, los identificadores se almacenan como texto aunque estén compuestos únicamente por números, porque no se utilizan para realizar cálculos matemáticos.


## 4.15 Validación final del ID

```python
df["user_id"].dtype
# Comprueba el tipo de dato.

df["user_id"].isna().sum()
# Cuenta los valores nulos.

df["user_id"].nunique()
# Cuenta los identificadores únicos, sin contar nulos por defecto.
```

Debemos confirmar que el tipo de dato es el adecuado, que los nulos están controlados y que la unicidad coincide con lo que esperamos de esa tabla.

## 4.16 Guardar el dataset limpio

Una vez finalizada la limpieza y las validaciones, podemos guardar el resultado en la carpeta `data/processed`.

```python
df.to_csv(
    "../data/processed/dataset_limpio.csv",
    index=False
)
```

- `to_csv()` exporta el DataFrame a un archivo CSV.
- `index=False` evita guardar el índice del DataFrame como una columna adicional.
- La carpeta `processed` debe existir antes de guardar el archivo.

**Buenas prácticas:** conservamos los datos originales en `data/raw` y guardamos los datos procesados en `data/processed`. Así mantenemos separado el origen de los datos del resultado de nuestro trabajo de limpieza.